# Central Texas Watershed Poster

**36×48" museum-scale poster** — Colorado River at Austin · Brazos River at Waco · Guadalupe River at Victoria.
20 analytical panels with descriptive labels and full data-source attribution.

In [ ]:
# ── imports & data ───────────────────────────────────────────────────────
import sys
from pathlib import Path

REPO = Path(".").resolve().parent
sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as ticker
import textwrap

from src import flow_metrics as fm
from tools.nwis_gauge import GaugeProvider
from tools.climate_index import ClimateIndexProvider
from tools.monthly_flow import MONTH_ABBR

NAS = "/Volumes/home/data/hydro-art"
OUT = REPO / "output" / "prints"
OUT.mkdir(parents=True, exist_ok=True)

%matplotlib inline

# ── theme ──
BG      = "#07080c"
PANEL   = "#10121b"
EDGE    = "#232838"
TEXT    = "#e6ebf5"
TICK    = "#8891a8"
CYAN    = "#00ffff"
MAGENTA = "#ff4d9a"
LIME    = "#00ff9c"
VIOLET  = "#9d00ff"
AMBER   = "#ff9c3a"
RED     = "#ff4d5e"
COOL_BL = "#00e5ff"
MUTED   = "#8891a8"
CAPTION = "#9aa3b8"   # slightly brighter for readability of captions

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": EDGE, "axes.labelcolor": TEXT,
    "text.color": TEXT, "xtick.color": TICK, "ytick.color": TICK,
    "figure.dpi": 100, "font.size": 9,
})

def complete_years(obs):
    return {y: v for y, v in obs.items() if np.all(np.isfinite(v))}

def sty(ax):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values(): s.set_color(EDGE)
    ax.tick_params(colors=TICK, labelsize=6)

# ── load data ──
colorado = GaugeProvider("08158000", NAS)
colorado_obs = colorado.monthly_means(1960, 2024)
brazos = GaugeProvider("08096500", NAS)
brazos_obs = brazos.monthly_means(1960, 2024)
guadalupe = GaugeProvider("08176500", NAS)
guadalupe_obs = guadalupe.monthly_means(1940, 2024)

oni = ClimateIndexProvider("oni", NAS).index_by_year(1940, 2024)
pdo = ClimateIndexProvider("pdo", NAS).index_by_year(1940, 2024)

colorado_c = complete_years(colorado_obs)
brazos_c   = complete_years(brazos_obs)
guadalupe_c = complete_years(guadalupe_obs)

RIVERS = [
    ("Colorado Rv at Austin",    colorado_c,  CYAN),
    ("Brazos Rv at Waco",        brazos_c,    MAGENTA),
    ("Guadalupe Rv at Victoria", guadalupe_c, LIME),
]

print(f"Colorado:  {len(colorado_c)} yr ({min(colorado_c)}–{max(colorado_c)})")
print(f"Brazos:    {len(brazos_c)} yr ({min(brazos_c)}–{max(brazos_c)})")
print(f"Guadalupe: {len(guadalupe_c)} yr ({min(guadalupe_c)}–{max(guadalupe_c)})")
print("ready")

---
## Compose the Poster

**Layout:** 36×48" at 150 DPI = 5400×7200 px. Grid: 6 columns × 16 rows.

Each graphic panel is 2 grid rows tall; each caption row is 1 row tall.
Panels are paired: graphic + caption text below it.

- Row 0: Title banner
- Rows 1–2: Hydrograph spaghetti (3 across) + Row 3: caption
- Rows 4–5: Typical year (3 across) + Row 6: caption
- Rows 7–8: Long-term trend (3 across) + Row 9: caption
- Rows 10–11: Low flow + Peak flow + ENSO composites + Row 12: caption
- Rows 13–14: FDC + Anomaly stripes + Heatmap + Row 15: caption
- Rows 16–17: Flashiness + Autocorrelation + Cumulative + Row 18: caption
- Rows 19–20: Seasonal ratio + Box plots + Rolling normals + Row 21: caption
- Rows 22–23: Record book + Decadal shift + Analog years + Row 24: caption
- Rows 25–26: Timing drift + Climate correlation + Cross-analysis + Row 27: caption
- Row 28: Data Sources & Attribution footer

In [ ]:
m = range(1, 13)
neon_cmap = LinearSegmentedColormap.from_list(
    "neon", ["#07080c", "#0a1628", "#002b4d", "#005577", "#00aaaa",
             "#00ffcc", "#66ff99", "#ffff66", "#ff9933", "#ff3366"])
cmap_div = LinearSegmentedColormap.from_list("anom", [COOL_BL, "#10121b", RED])

# ── caption helper ──────────────────────────────────────────────────────
def cap(ax, title, body, *, width=52):
    """Render a styled caption into a blank axes: bold title + wrapped body."""
    ax.set_facecolor(BG)
    ax.set_axis_off()
    wrapped = "\n".join(textwrap.wrap(body, width=width))
    ax.text(0.03, 0.92, title, transform=ax.transAxes, va="top",
            fontsize=7, fontweight="bold", color=TEXT, fontfamily="sans-serif")
    ax.text(0.03, 0.72, wrapped, transform=ax.transAxes, va="top",
            fontsize=5.5, color=CAPTION, fontfamily="sans-serif", linespacing=1.35)

# ── simplified Texas outline (lon, lat) ─────────────────────────────────
TX_OUTLINE = [
    (-106.65, 31.75), (-106.50, 32.00), (-103.06, 32.00), (-103.06, 36.50),
    (-100.00, 36.50), (-100.00, 34.56), (-99.20, 34.20), (-98.10, 34.00),
    (-96.50, 33.85), (-95.30, 33.95), (-94.48, 33.60), (-94.04, 33.55),
    (-94.00, 33.00), (-93.85, 31.80), (-93.55, 30.20), (-93.82, 29.77),
    (-94.70, 29.35), (-95.10, 28.95), (-96.20, 28.45), (-97.00, 27.80),
    (-97.15, 26.40), (-97.40, 25.84), (-97.70, 26.00), (-98.30, 26.20),
    (-99.10, 26.42), (-99.70, 27.00), (-100.10, 28.10), (-100.67, 29.10),
    (-101.40, 29.77), (-102.30, 29.80), (-103.00, 29.00), (-104.00, 29.30),
    (-104.70, 29.60), (-105.40, 30.60), (-106.20, 31.40), (-106.65, 31.75),
]
TX_LON = [p[0] for p in TX_OUTLINE]
TX_LAT = [p[1] for p in TX_OUTLINE]

# Gauge locations (lon, lat)
GAUGES = {
    "Colorado Rv\nat Austin":    (-97.69, 30.28, CYAN),
    "Brazos Rv\nat Waco":        (-97.07, 31.52, MAGENTA),
    "Guadalupe Rv\nat Victoria":  (-97.01, 28.79, LIME),
}

# ════════════════════════════════════════════════════════════════════════
#  BUILD THE POSTER — every panel individually labeled
# ════════════════════════════════════════════════════════════════════════
ROWS = 30
fig = plt.figure(figsize=(36, 48), facecolor=BG, dpi=150)
gs = GridSpec(ROWS, 6, figure=fig, hspace=0.12, wspace=0.20,
              left=0.03, right=0.97, top=0.975, bottom=0.005,
              height_ratios=(
                  [2.2]            # 0  Texas graphic + title
                + [0.45]           # 1  subtitle / info line
                + [1, 1, 0.55]     # 2-4
                + [1, 1, 0.55]     # 5-7
                + [1, 1, 0.55]     # 8-10
                + [1, 1, 0.55]     # 11-13
                + [1, 1, 0.55]     # 14-16
                + [1, 1, 0.55]     # 17-19
                + [1, 1, 0.55]     # 20-22
                + [1, 1, 0.55]     # 23-25
                + [1, 1, 0.55]     # 26-28
                + [0.85]           # 29 footer
              ))

# ═══════════════════ ROW 0: TEXAS GRAPHIC + TITLE ═════════════════════
ax_tx = fig.add_subplot(gs[0, 0:3])
ax_tx.set_facecolor(BG)
ax_tx.fill(TX_LON, TX_LAT, facecolor="#0d1220", edgecolor=CYAN, linewidth=2.5)
for label, (lon, lat, color) in GAUGES.items():
    ax_tx.plot(lon, lat, "o", color=color, ms=12, markeredgecolor="white",
               markeredgewidth=1.5, zorder=5)
    ax_tx.annotate(label, (lon, lat), textcoords="offset points",
                   xytext=(14, -4), fontsize=8, color=color, fontweight="bold",
                   fontfamily="sans-serif",
                   arrowprops=dict(arrowstyle="-", color=color, lw=0.8))
ax_tx.set_xlim(-107.5, -92.5)
ax_tx.set_ylim(25.0, 37.5)
ax_tx.set_aspect("equal")
ax_tx.set_axis_off()

ax_title = fig.add_subplot(gs[0, 3:6])
ax_title.set_facecolor(BG); ax_title.set_axis_off()
ax_title.text(0.50, 0.75, "CENTRAL\nTEXAS\nWATERSHEDS", ha="center", va="center",
              fontsize=56, fontweight="bold", color=CYAN, fontfamily="monospace",
              transform=ax_title.transAxes, linespacing=1.15)
ax_title.text(0.50, 0.22, "27 Analytical Panels\n1940–2024",
              ha="center", va="center", fontsize=16, color=MUTED,
              fontfamily="monospace", transform=ax_title.transAxes, linespacing=1.4)

# ═══════════════════ ROW 1: SUBTITLE BAR ══════════════════════════════
ax_sub = fig.add_subplot(gs[1, :]); ax_sub.set_facecolor(BG); ax_sub.set_axis_off()
ax_sub.text(0.5, 0.5,
    "COLORADO RIVER AT AUSTIN  ·  BRAZOS RIVER AT WACO  ·  GUADALUPE RIVER AT VICTORIA"
    "      |      USGS NWIS Daily Discharge  ·  NOAA ONI + PDO",
    ha="center", va="center", fontsize=13, color=TICK, fontfamily="monospace",
    transform=ax_sub.transAxes)

# ═══════════════════ ROWS 2-4: HYDROGRAPH SPAGHETTI ═══════════════════
for col, (name, obs, color) in enumerate(RIVERS):
    ax = fig.add_subplot(gs[2:4, col*2:(col+1)*2]); sty(ax)
    years = sorted(obs.keys())
    cmap_v = plt.get_cmap("viridis")
    span = max(years) - min(years) or 1
    for y in years:
        c = cmap_v((y - min(years)) / span)
        ax.plot(m, obs[y], color=c, lw=0.7, alpha=0.65)
    ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
    ax.set_ylabel("cfs", fontsize=7)
    ax.set_title(f"{name}  ({min(years)}–{max(years)})", fontsize=9, pad=4)
    sm = plt.cm.ScalarMappable(cmap="viridis",
         norm=plt.Normalize(vmin=min(years), vmax=max(years)))
    fig.colorbar(sm, ax=ax, label="year", shrink=0.7, pad=0.02)

for col, (name, _, _) in enumerate(RIVERS):
    ax_c = fig.add_subplot(gs[4, col*2:(col+1)*2])
    cap(ax_c, f"Hydrograph Spaghetti — {name.split()[0]}",
        "Every year's monthly flow as a single line (Jan–Dec), color-coded oldest (purple) "
        "to newest (yellow). Wide vertical spread = high variability for that month. "
        "If yellow lines cluster differently than purple, the seasonal pattern is shifting.")

# ═══════════════════ ROWS 5-7: TYPICAL YEAR ═══════════════════════════
for col, (name, obs, color) in enumerate(RIVERS):
    ax = fig.add_subplot(gs[5:7, col*2:(col+1)*2]); sty(ax)
    years = sorted(obs.keys())
    stack = np.array([obs[y] for y in years])
    mean = np.nanmean(stack, axis=0)
    p10, p90 = np.nanpercentile(stack, 10, axis=0), np.nanpercentile(stack, 90, axis=0)
    p25, p75 = np.nanpercentile(stack, 25, axis=0), np.nanpercentile(stack, 75, axis=0)
    ax.fill_between(m, p10, p90, color=color, alpha=0.08, label="P10–P90")
    ax.fill_between(m, p25, p75, color=color, alpha=0.15, label="P25–P75")
    ax.plot(m, mean, color=color, lw=2, label="mean")
    ax.plot(m, np.nanmedian(stack, axis=0), color=AMBER, lw=1.5, ls="--", label="median")
    cot = fm.center_of_timing(mean)
    ax.axvline(cot, color=AMBER, ls=":", lw=1, label=f"COT {cot:.1f}")
    ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
    ax.set_ylabel("cfs", fontsize=7)
    ax.set_title(f"{name} — Typical Year", fontsize=9, pad=4)
    ax.legend(fontsize=5, frameon=False)

for col, (name, _, _) in enumerate(RIVERS):
    ax_c = fig.add_subplot(gs[7, col*2:(col+1)*2])
    cap(ax_c, f"Typical Year — {name.split()[0]}",
        "Mean (solid) and median (dashed) monthly flow with P10–P90 and P25–P75 envelopes. "
        "Dotted line = Center of Timing (month by which 50% of flow has passed). "
        "When mean >> median, a few extreme floods skew the average upward.")

# ═══════════════════ ROWS 8-10: LONG-TERM TREND ══════════════════════
for col, (name, obs, color) in enumerate(RIVERS):
    ax = fig.add_subplot(gs[8:10, col*2:(col+1)*2]); sty(ax)
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    mk = fm.mann_kendall(annual)
    slope = fm.sens_slope(annual)
    x = np.arange(len(years))
    fit = np.median(annual) + slope * (x - np.median(x))
    ax.plot(years, annual, "o-", color=color, lw=1, ms=2.5, label="annual mean")
    ax.plot(years, fit, "--", color=VIOLET, lw=1.5, label=f"Sen {slope:+.1f} cfs/yr")
    if len(annual) >= 10:
        kernel = np.ones(10) / 10
        rolling = np.convolve(annual, kernel, mode="valid")
        ax.plot(years[4:4+len(rolling)], rolling, color=AMBER, lw=1.8, alpha=0.8, label="10-yr rolling")
    ax.set_ylabel("annual mean (cfs)", fontsize=7)
    ax.set_title(f"{name}\nMK: {mk.trend} (τ={mk.tau:+.3f}, p={mk.p:.4f})", fontsize=8, pad=4)
    ax.legend(fontsize=5, frameon=False)

for col, (name, obs, _) in enumerate(RIVERS):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    mk = fm.mann_kendall(annual)
    ax_c = fig.add_subplot(gs[10, col*2:(col+1)*2])
    cap(ax_c, f"Long-Term Trend — {name.split()[0]}",
        f"Each dot = one year's mean flow. Purple dashed = Sen's slope (outlier-resistant). "
        f"Amber = 10-yr rolling mean. Mann-Kendall: τ={mk.tau:+.3f}, p={mk.p:.4f}. "
        f"'No trend' with high p means variability dominates over any directional shift.")

# ═══════════════════ ROWS 11-13: LOW FLOW + PEAK + ENSO ══════════════
ax = fig.add_subplot(gs[11:13, 0:2]); sty(ax)
for name, obs, color in RIVERS:
    years = sorted(obs.keys())
    low = np.array([float(np.nanmin(obs[y][5:9])) for y in years])
    slope = fm.sens_slope(low)
    x = np.arange(len(years))
    fit = np.median(low) + slope * (x - np.median(x))
    ax.plot(years, low, "o-", color=color, lw=0.8, ms=2, alpha=0.7, label=name.split()[0])
    ax.plot(years, fit, "--", color=color, lw=0.8, alpha=0.4)
ax.set_ylabel("summer-low cfs", fontsize=7)
ax.set_title("Summer Low Flow (Jun–Sep)", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False)

ax_c = fig.add_subplot(gs[13, 0:2])
cap(ax_c, "Summer Low Flow — All Rivers",
    "Lowest monthly-mean during Jun–Sep each year — the critical "
    "ecological and water-supply metric. Declining trend = growing "
    "drought stress. Years near zero = river approached intermittent "
    "conditions. Dams regulate baseflow on the Colorado at Austin.")

ax = fig.add_subplot(gs[11:13, 2:4]); sty(ax)
for name, obs, color in RIVERS:
    years = sorted(obs.keys())
    peaks = np.array([float(np.nanmax(obs[y])) for y in years])
    slope = fm.sens_slope(peaks)
    x = np.arange(len(years))
    fit = np.median(peaks) + slope * (x - np.median(x))
    ax.plot(years, peaks, "o-", color=color, lw=0.8, ms=2, alpha=0.7, label=name.split()[0])
    ax.plot(years, fit, "--", color=color, lw=0.8, alpha=0.4)
ax.set_ylabel("peak monthly cfs", fontsize=7)
ax.set_title("Annual Peak Flow", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False)

ax_c = fig.add_subplot(gs[13, 2:4])
cap(ax_c, "Annual Peak Flow — All Rivers",
    "Highest monthly-mean each year — tracing flood intensity over "
    "decades. Upward trend = storms getting bigger. Isolated spikes "
    "far above the line are landmark flood events (tropical remnants, "
    "stalled fronts, training thunderstorms).")

ax = fig.add_subplot(gs[11:13, 4:6]); sty(ax)
comp = fm.composite_hydrographs(brazos_c, oni, warm_min=0.5, cool_max=-0.5)
ax.plot(m, comp.warm, color=RED, lw=2, label=f"El Niño (n={len(comp.warm_years)})")
ax.plot(m, comp.cool, color=COOL_BL, lw=2, label=f"La Niña (n={len(comp.cool_years)})")
ax.plot(m, comp.neutral, color=MUTED, lw=1.5, ls="--", label=f"Neutral (n={len(comp.neutral_years)})")
ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("cfs", fontsize=7)
ax.set_title("Brazos — ENSO Composite", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

ax_c = fig.add_subplot(gs[13, 4:6])
cap(ax_c, "ENSO Composite — Brazos",
    "Average hydrograph during El Niño (red, ONI > +0.5), La Niña "
    "(blue, ONI < −0.5), and neutral years. In Texas, El Niño winters "
    "are typically wetter — red above blue in Oct–Mar confirms the "
    "teleconnection. Curve separation = signal strength.")

# ═══════════════════ ROWS 14-16: FDC + ANOMALY + HEATMAP ═════════════
ax = fig.add_subplot(gs[14:16, 0:2]); sty(ax)
quantiles = (0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99)
decades = fm.decade_flow_duration(brazos_c, quantiles)
cmap_p = plt.get_cmap("plasma")
n_dec = max(1, len(decades) - 1)
for j, d in enumerate(decades):
    ax.semilogy([q * 100 for q in d.quantiles], d.flows,
                "o-", color=cmap_p(j / n_dec), lw=1.2, ms=2, label=f"{d.decade}s")
ax.set_xlabel("exceedance %", fontsize=7); ax.set_ylabel("cfs", fontsize=7)
ax.set_title("Brazos — Flow Duration by Decade", fontsize=9, pad=4)
ax.legend(fontsize=4, frameon=False, ncol=2)

ax_c = fig.add_subplot(gs[16, 0:2])
cap(ax_c, "Flow Duration Curves — Brazos",
    "FDC answers 'what flow is exceeded X% of the time?' — one curve "
    "per decade, log scale. Left side = extreme highs; right = baseflow. "
    "Decades shifting up = wetter. Parallel curves = regime scaled "
    "proportionally; crossing = highs and lows moved differently.")

ax = fig.add_subplot(gs[14:16, 2:4]); sty(ax)
for ri, (name, obs, color) in enumerate(RIVERS):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    normal = np.mean(annual)
    anomalies = annual - normal
    vmax = max(abs(anomalies.min()), abs(anomalies.max()))
    for y_val, a in zip(years, anomalies):
        c = cmap_div((a / vmax + 1) / 2)
        ax.bar(y_val, 0.28, bottom=ri * 0.32, color=c, width=1.0, edgecolor="none")
ax.set_yticks([i * 0.32 + 0.14 for i in range(3)])
ax.set_yticklabels([n.split()[0] for n, _, _ in RIVERS], fontsize=6)
ax.set_title("Annual Anomaly Stripes", fontsize=9, pad=4)

ax_c = fig.add_subplot(gs[16, 2:4])
cap(ax_c, "Anomaly Stripes — All Rivers",
    "Each bar = one year's departure from the long-term mean. Cyan = "
    "wetter than average; red = drier. Multi-year runs of the same "
    "color reveal persistent drought or wet epochs. When all three "
    "rivers match, a large-scale climate driver is at work.")

ax = fig.add_subplot(gs[14:16, 4:6]); sty(ax)
years_g = sorted(guadalupe_c.keys())
mat = np.array([guadalupe_c[y] for y in years_g])
im = ax.imshow(mat, aspect="auto", cmap=neon_cmap, interpolation="nearest",
               extent=[0.5, 12.5, years_g[-1]+0.5, years_g[0]-0.5])
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("year", fontsize=7)
ax.set_title("Guadalupe — Monthly Flow Heatmap", fontsize=9, pad=4)
fig.colorbar(im, ax=ax, label="cfs", shrink=0.7, pad=0.02)

ax_c = fig.add_subplot(gs[16, 4:6])
cap(ax_c, "Monthly Heatmap — Guadalupe",
    "Year × month grid: color intensity = flow magnitude. Dark blue = "
    "high flow; pale = low. Vertical streaks = a consistently wet "
    "month. Horizontal streaks = drought years. Isolated bright spots "
    "= individual flood events (e.g. 2015 Memorial Day floods).")

# ═══════════════════ ROWS 17-19: FLASHINESS + AUTOCORR + CUMULATIVE ══
ax = fig.add_subplot(gs[17:19, 0:2]); sty(ax)
for name, obs, color in RIVERS:
    years = sorted(obs.keys())
    fi = [fm.flashiness(obs[y]) for y in years]
    ax.plot(years, fi, "o-", color=color, lw=0.8, ms=2, alpha=0.7, label=name.split()[0])
ax.set_ylabel("flashiness index", fontsize=7)
ax.set_title("Richards-Baker Flashiness", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False)

ax_c = fig.add_subplot(gs[19, 0:2])
cap(ax_c, "Flashiness Index — All Rivers",
    "Richards-Baker Index: ratio of month-to-month flow changes to "
    "total flow. Higher = more abrupt swings between wet and dry. "
    "Rising flashiness may indicate intensifying storms, more "
    "impervious surface, or declining baseflow.")

ax = fig.add_subplot(gs[17:19, 2:4]); sty(ax)
years_b = sorted(brazos_c.keys())
annual_b = np.array([float(np.nanmean(brazos_c[y])) for y in years_b])
mean_b, std_b = np.mean(annual_b), np.std(annual_b)
n_b = len(annual_b)
acf = []
for lag in range(16):
    if lag >= n_b: acf.append(0.0); continue
    c = np.mean((annual_b[:n_b-lag] - mean_b) * (annual_b[lag:] - mean_b)) / (std_b**2)
    acf.append(c)
ax.bar(range(16), acf, color=MAGENTA, alpha=0.7, width=0.6)
ci = 1.96 / np.sqrt(n_b)
ax.axhline(ci, color=AMBER, ls="--", lw=1, label=f"95% CI (±{ci:.2f})")
ax.axhline(-ci, color=AMBER, ls="--", lw=1)
ax.axhline(0, color=MUTED, lw=0.5)
ax.set_xlabel("lag (years)", fontsize=7); ax.set_ylabel("autocorrelation", fontsize=7)
ax.set_title("Brazos — Autocorrelation", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

ax_c = fig.add_subplot(gs[19, 2:4])
cap(ax_c, "Autocorrelation — Brazos",
    "Correlation of annual flow with itself at 1–15 year lags. Bars "
    "above the amber 95% CI dashed line are statistically significant. "
    "Positive at lag 1 = wet years cluster (persistence). Significant "
    "bars at 2–5 yr lags suggest ENSO-scale cycling.")

ax = fig.add_subplot(gs[17:19, 4:6]); sty(ax)
for name, obs, color in RIVERS:
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    mean = np.mean(annual)
    cumulative = np.cumsum(annual - mean)
    ax.plot(years, cumulative, color=color, lw=1.8, label=name.split()[0])
    ax.fill_between(years, 0, cumulative, alpha=0.08, color=color)
ax.axhline(0, color=MUTED, lw=0.5, ls=":")
ax.set_ylabel("cumulative departure (cfs)", fontsize=7)
ax.set_title("Cumulative Departure", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False)

ax_c = fig.add_subplot(gs[19, 4:6])
cap(ax_c, "Cumulative Departure — All Rivers",
    "Running total of above/below-average flow years — a 'water "
    "debt/surplus' ledger. Rising slopes = wet epoch; falling = drought. "
    "Peaks mark end of wet periods; troughs = drought bottoms. The "
    "deepest trough is the worst cumulative drought on record.")

# ═══════════════════ ROWS 20-22: RATIO + BOX + NORMALS ═══════════════
ax = fig.add_subplot(gs[20:22, 0:2]); sty(ax)
for name, obs, color in RIVERS:
    years = sorted(obs.keys())
    ratios = [fm.seasonal_ratio(obs[y], wet=(10, 11, 12, 1, 2, 3, 4)) for y in years]
    ax.plot(years, ratios, "o-", color=color, lw=0.8, ms=2, alpha=0.7, label=name.split()[0])
ax.axhline(0.5, color=MUTED, ls=":", lw=0.8, alpha=0.5)
ax.set_ylabel("wet-season fraction", fontsize=7)
ax.set_title("Seasonal Ratio (Oct–Apr / Total)", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False)

ax_c = fig.add_subplot(gs[22, 0:2])
cap(ax_c, "Seasonal Ratio — All Rivers",
    "Fraction of annual flow arriving in the wet season (Oct–Apr). "
    "Above 0.5 = wet season dominates; below = summer storms contribute "
    "more. Rising trend = flow concentrating into winter (drier summers). "
    "High scatter is normal — one tropical event can shift the ratio.")

ax = fig.add_subplot(gs[20:22, 2:4]); sty(ax)
years_co = sorted(colorado_c.keys())
data = [[colorado_c[y][mi] for y in years_co] for mi in range(12)]
bp = ax.boxplot(data, positions=range(1, 13), widths=0.6, patch_artist=True,
                showfliers=True, flierprops=dict(marker="o", markersize=2,
                markerfacecolor=RED, markeredgecolor=RED, alpha=0.5))
for patch in bp["boxes"]:
    patch.set_facecolor(CYAN); patch.set_alpha(0.3)
for el in ("whiskers", "caps", "medians"):
    for line in bp[el]: line.set_color(TEXT)
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("cfs", fontsize=7)
ax.set_title("Colorado — Monthly Distribution", fontsize=9, pad=4)

ax_c = fig.add_subplot(gs[22, 2:4])
cap(ax_c, "Monthly Distribution — Colorado",
    "Box = interquartile range (25th–75th %ile), white line = median, "
    "whiskers = 1.5×IQR, red dots = outliers beyond. Tall boxes = high "
    "variability. Many outliers above whiskers = occasional extreme "
    "floods. Median near box bottom = right-skewed (flood-heavy).")

ax = fig.add_subplot(gs[20:22, 4:6]); sty(ax)
for name, obs, color in RIVERS:
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    normals = fm.rolling_normals(annual, window=30)
    if normals:
        centers = [years[0] + (n.start + n.end) // 2 for n in normals]
        values = [n.mean for n in normals]
        ax.plot(centers, values, color=color, lw=2.5, label=name.split()[0])
        ax.fill_between(centers, values, alpha=0.08, color=color)
ax.set_ylabel("cfs", fontsize=7)
ax.set_title("30-Year Rolling Normals", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False)

ax_c = fig.add_subplot(gs[22, 4:6])
cap(ax_c, "Rolling Normals — All Rivers",
    "30-year rolling average of annual flow — the 'climate normal' — "
    "and how it has shifted. Rising = multi-decadal wet phase; dipping "
    "= dry. When all three rivers rise/fall together, it is a regional "
    "climate signal. Divergence = local factors (dams, urbanization).")

# ═══════════════════ ROWS 23-25: RECORDS + DECADAL + ANALOG ══════════
ax = fig.add_subplot(gs[23:25, 0:2]); sty(ax); ax.set_axis_off()
lines = []
for name, obs, color in RIVERS:
    rb = fm.record_book(obs, n=3)
    short = name.split()[0]
    lines.append(f"  {'─'*30}")
    lines.append(f"  {short.upper()}")
    lines.append(f"  Wettest Years:")
    for e in rb.wettest_years:
        lines.append(f"    #{e.rank}  {e.year}  {e.value:>8,.0f} cfs")
    lines.append(f"  Driest Summers:")
    for e in rb.driest_summers:
        lines.append(f"    #{e.rank}  {e.year}  {e.value:>8,.0f} cfs")
    lines.append("")
ax.text(0.05, 0.95, "\n".join(lines), transform=ax.transAxes, va="top",
        family="monospace", fontsize=7, color=TEXT)
ax.set_title("Record Book", fontsize=9, pad=4, color=TEXT)

ax_c = fig.add_subplot(gs[25, 0:2])
cap(ax_c, "Record Book — All Rivers",
    "All-time top 3 wettest years and driest summers for each river. "
    "If recent years dominate, extremes are intensifying. If old years "
    "dominate, the modern record may be more regulated. Same years "
    "across rivers = region-wide event (e.g. 2011 drought).")

ax = fig.add_subplot(gs[23:25, 2:4]); sty(ax)
for name, obs, color in RIVERS:
    years = sorted(obs.keys())
    mid = years[len(years) // 2]
    early_mean = np.mean([obs[y] for y in years if y < mid], axis=0)
    late_mean = np.mean([obs[y] for y in years if y >= mid], axis=0)
    diff = late_mean - early_mean
    ax.plot(m, diff, "o-", color=color, lw=1.5, ms=3, label=name.split()[0])
ax.axhline(0, color=MUTED, lw=0.8, ls=":")
ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("Δ cfs (late − early)", fontsize=7)
ax.set_title("Decadal Shift (late half − early half)", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False)

ax_c = fig.add_subplot(gs[25, 2:4])
cap(ax_c, "Decadal Shift — All Rivers",
    "Difference between late-period and early-period mean monthly flow. "
    "Positive = that month got wetter over time; negative = drier. "
    "Crossing zero = the seasonal shape changed — peak flow may have "
    "shifted to a different month. A simple regime-change detector.")

ax = fig.add_subplot(gs[23:25, 4:6]); sty(ax)
target_g = max(guadalupe_c.keys())
analogs = fm.analog_years(guadalupe_c, target_g, n=8)
ax.plot(m, guadalupe_c[target_g], color=LIME, lw=3, label=f"{target_g} (target)")
cmap_a = plt.get_cmap("plasma")
for i, a in enumerate(analogs):
    ax.plot(m, guadalupe_c[a.year], color=cmap_a(i / 8), lw=1, alpha=0.7,
            label=f"{a.year} ({a.similarity:.2f})")
ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("cfs", fontsize=7)
ax.set_title(f"Guadalupe — Analog Years to {target_g}", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False, ncol=2)

ax_c = fig.add_subplot(gs[25, 4:6])
cap(ax_c, f"Analog Years — Guadalupe ({target_g})",
    "Most recent year (bold) vs. its 8 closest historical matches by "
    "cosine similarity. Score 0–1 in legend (>0.95 = very similar). "
    "If analogs cluster in one decade, it suggests recurring climate "
    "patterns. Used in seasonal forecasting: 'what happened next?'")

# ═══════════════════ ROWS 26-28: TIMING + CLIMATE + CROSS ════════════
ax = fig.add_subplot(gs[26:28, 0:2]); sty(ax)
for name, obs, color in RIVERS:
    years = sorted(obs.keys())
    cot_vals = [fm.center_of_timing(obs[y]) for y in years]
    ax.plot(years, cot_vals, "o-", color=color, lw=0.8, ms=2, alpha=0.7, label=name.split()[0])
ax.set_yticks(range(1, 13)); ax.set_yticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("center of timing", fontsize=7)
ax.set_title("Center-of-Timing Drift", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False)

ax_c = fig.add_subplot(gs[28, 0:2])
cap(ax_c, "Timing Drift — All Rivers",
    "Center of Timing each year — the fractional month by which 50% of "
    "flow has passed. Persistent drift signals changing storm seasonality. "
    "In rain-dominated Texas basins this is noisy — a single hurricane "
    "can shift it — but a trend over decades is meaningful.")

ax = fig.add_subplot(gs[26:28, 2:4]); sty(ax); ax.set_axis_off()
lines = ["  CLIMATE CORRELATION", "  ─────────────────────────────────────────────",
         f"  {'River':<14} {'ONI→Ann':>9} {'ONI→Pk':>9} {'PDO→Ann':>9} {'PDO→Pk':>9}",
         "  " + "─" * 54]
for name, obs, _ in RIVERS:
    years = sorted(obs.keys())
    ann = {y: float(np.nanmean(obs[y])) for y in years}
    pk = {y: float(np.nanmax(obs[y])) for y in years}
    r_oa = fm.correlate(ann, oni); r_op = fm.correlate(pk, oni)
    r_pa = fm.correlate(ann, pdo); r_pp = fm.correlate(pk, pdo)
    short = name.split()[0]
    lines.append(f"  {short:<14} {r_oa:>+7.3f}   {r_op:>+7.3f}   {r_pa:>+7.3f}   {r_pp:>+7.3f}")
lines += ["", "  |r| > 0.3 suggests meaningful teleconnection"]
ax.text(0.05, 0.90, "\n".join(lines), transform=ax.transAxes, va="top",
        family="monospace", fontsize=7, color=TEXT)
ax.set_title("Climate Teleconnections", fontsize=9, pad=4, color=TEXT)

ax_c = fig.add_subplot(gs[28, 2:4])
cap(ax_c, "Climate Teleconnections — All Rivers",
    "Pearson r between ONI (ENSO) / PDO and annual mean / peak flow. "
    "Positive r with ONI = El Niño brings more water (expected for TX). "
    "|r| > 0.3 = exploitable for forecasting. Weak r = that climate "
    "index doesn't predict flow for that basin.")

ax = fig.add_subplot(gs[26:28, 4:6]); sty(ax)
for name, obs, color in RIVERS:
    years = sorted(obs.keys())
    pk_by_year = {y: float(np.nanmax(obs[y])) for y in years}
    metric, index = fm.align_index(pk_by_year, oni)
    ax.scatter(index, metric, s=14, c=color, alpha=0.5, edgecolors="none",
               label=name.split()[0])
ax.set_xlabel("ONI (ENSO)", fontsize=7); ax.set_ylabel("peak flow (cfs)", fontsize=7)
ax.set_title("Peak Flow vs ONI", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False)

ax_c = fig.add_subplot(gs[28, 4:6])
cap(ax_c, "Peak Flow vs ENSO — All Rivers",
    "Scatter of each year's peak monthly flow against that year's ONI "
    "value. Upward slope from left to right = El Niño amplifies floods. "
    "Tight cloud = weak relationship; spread with slope = real signal. "
    "Visual confirmation of the correlation table to its left.")

# ═══════════════════ ROW 29: DATA SOURCES FOOTER ═════════════════════
ax = fig.add_subplot(gs[29, :]); ax.set_facecolor(BG); ax.set_axis_off()

footer = (
    "DATA SOURCES & ATTRIBUTION\n"
    "─────────────────────────────────────────────────────────────────────────────────────────────────────────────────\n"
    "Daily mean discharge (param 00060, stat 00003): USGS National Water Information System (NWIS)  ·  waterservices.usgs.gov/nwis/dv/\n"
    "ENSO — Oceanic Niño Index (ONI): NOAA Climate Prediction Center (CPC)  ·  cpc.ncep.noaa.gov/data/indices/oni.ascii.txt\n"
    "Pacific Decadal Oscillation (PDO): NOAA NCEI ERSST v5  ·  ncei.noaa.gov/pub/data/cmb/ersst/v5/index/ersst.v5.pdo.dat\n"
    "─────────────────────────────────────────────────────────────────────────────────────────────────────────────────\n"
    "USGS GAUGES:  08158000 Colorado Rv at Austin (1960–2024)  ·  08096500 Brazos Rv at Waco (1960–2024)  ·  08176500 Guadalupe Rv at Victoria (1940–2024)\n"
    "─────────────────────────────────────────────────────────────────────────────────────────────────────────────────\n"
    "METHODS:  Mann-Kendall non-parametric trend test  ·  Sen's robust slope estimator  ·  Center of Timing (flow-weighted mean month)\n"
    "Richards-Baker Flashiness Index  ·  Decade-stratified Flow Duration Curves  ·  ENSO Composite Hydrographs (ONI ±0.5 threshold)\n"
    "Cosine-similarity Analog Years  ·  30-year Rolling Climate Normals  ·  Cumulative Departure Analysis\n"
    "─────────────────────────────────────────────────────────────────────────────────────────────────────────────────\n"
    "All source data are U.S. federal public domain.  ·  Metrics computed by src/flow_metrics.py (deterministic, offline, numpy-only).\n"
    "Raw NWIS and climate-index data snapshotted locally at fetch time for reproducibility.  ·  hydro-art watershed report toolchain."
)

ax.text(0.5, 0.95, footer, transform=ax.transAxes, va="top", ha="center",
        fontsize=7.5, color=CAPTION, fontfamily="monospace", linespacing=1.5)

# ═══════════════════ SAVE ═════════════════════════════════════════════
out_path = OUT / "central_texas_poster.png"
fig.savefig(out_path, facecolor=BG, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")